In [1]:
# Parameters
BATCH_MODE = "true"


# SC-25 : Deploiement Mainnet (L2)

**Navigation** : [Sommaire](../README.md) | [<< Précédent](SC-24-Testnet-Deploy-Python.ipynb) | [Suivant >>](SC-26-Final-Project-Python.ipynb)

***

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Choisir un **L2** (Base, Arbitrum) ou une **sidechain EVM** (Polygon PoS) en fonction du cout, de la vitesse et du modele de securite
2. **Estimer le cout** d'un deploiement en ETH et en USD, a partir d'un gas price mesure sur chaque chaine
3. **Deployer** un contrat sur un L2 mainnet avec du gas reel
4. Appliquer la **checklist de securite** avant un deploiement production
5. **Verifier** un contrat sur l'explorateur de blocs

### Prerequis

- [SC-24-Testnet-Deploy-Python](SC-24-Testnet-Deploy-Python.ipynb) complete
- ETH sur Base ou Polygon (quelques centimes suffisent pour un deploiement simple)
- `pip install web3 py-solc-x eth-account python-dotenv`

### Duree estimee : 40 minutes

> **Cout de deploiement : il est mesure en section 1**, sur trois chaines a l'instant de l'execution. Les montants de ce carnet sont donc des ordres de grandeur temporels, pas des constantes.

***

## 0. Pourquoi un L2 ?

Le cout d'un deploiement depend du prix du gas de la chaine, et ce prix change d'un jour a l'autre. Plutot que d'avancer des montants figes, la section 1 **mesure** le gas price de chaque chaine au moment de l'execution, et en deduit le cout d'un deploiement simple (300 000 gas).

Deux familles de reseaux, deux modeles de securite :

| Reseau | Modele | D'ou vient la securite |
|--------|--------|--------------------------|
| Ethereum L1 | chaine principale | reference : tout le monde s'y rattache |
| Base | rollup optimiste | bornee par Ethereum L1 (transactions publiees et prouvees sur L1) |
| Arbitrum | rollup optimiste | bornee par Ethereum L1 |
| Polygon PoS | **sidechain** EVM | **son propre** jeu de validateurs : elle n'herite pas de la securite d'Ethereum |

Un attaquant qui devrait briser Ethereum L1 pour reecrire l'historique de Base n'a pas les memes moyens que sur Polygon PoS, dont le comite de validateurs est autonome. Choisir « un L2 » sans distinguer ces deux familles, c'est se tromper sur la seule question qui compte quand des fonds sont en jeu : **qui repond du tresorerie ?**

***

## 1. Configuration Base (L2 de Coinbase)

In [2]:
import os
from dotenv import load_dotenv

try:
    from web3 import Web3
    import solcx
    WEB3_AVAILABLE = True
except ImportError:
    WEB3_AVAILABLE = False
    print("web3 ou py-solc-x non installe. Installez avec: pip install web3 py-solc-x")

load_dotenv()

# Base mainnet
BASE_RPC = os.getenv("BASE_RPC", "https://mainnet.base.org")
PRIVATE_KEY = os.getenv("DEPLOYER_PRIVATE_KEY", "")

# Chaines comparees : le cout d'un deploiement se lit dans le prix du gas de chaque chaine
CHAINS = {
    "Base": BASE_RPC,
    "Polygon": os.getenv("POLYGON_RPC", "https://polygon-bor-rpc.publicnode.com"),
    "Ethereum L1": os.getenv("ETH_RPC", "https://ethereum-rpc.publicnode.com"),
}
GAS_DEPLOY = 300000            # gas pour un contrat simple
PRIX_ETH_USD = 2500.0          # hypothese de travail : le carnet ne lit pas le prix de l'ETH
PRIX_ETH_PAR_CHAINE = {"Base": PRIX_ETH_USD, "Polygon": 0.50, "Ethereum L1": PRIX_ETH_USD}

if WEB3_AVAILABLE:
    w3 = Web3(Web3.HTTPProvider(BASE_RPC))

    if w3.is_connected():
        chain_id = w3.eth.chain_id  # 8453 pour Base
        gas_price = w3.eth.gas_price
        print(f"Connecte a Base (chain_id={chain_id})")
        print(f"Gas price: {w3.from_wei(gas_price, 'gwei'):.4f} gwei")
        print(f"Bloc: {w3.eth.block_number:,}")
    else:
        print("Connexion echouee. Base RPC public peut etre lent.")
        print("Alternative: utilisez Alchemy/Infura avec une cle API Base")
else:
    print("Section sautee : web3 non disponible")

# Comparaison des couts : une mesure par chaine, a l'instant de l'execution
if WEB3_AVAILABLE:
    print(f"\nCout d'un deploiement simple ({GAS_DEPLOY:,} gas), par chaine :")
    for nom, rpc in CHAINS.items():
        try:
            w_chaine = Web3(Web3.HTTPProvider(rpc, request_kwargs={"timeout": 20}))
            if not w_chaine.is_connected():
                raise ConnectionError("RPC injoignable")
            gp = w_chaine.eth.gas_price
            cout_eth = float(w_chaine.from_wei(GAS_DEPLOY * gp, "ether"))
            print(f"  {nom}: chain_id={w_chaine.eth.chain_id}, gas={w_chaine.from_wei(gp, 'gwei'):.4f} gwei, "
                  f"cout={cout_eth:.10f} ETH (~${cout_eth * PRIX_ETH_PAR_CHAINE[nom]:.4f})")
        except Exception as e:
            print(f"  {nom}: mesure indisponible ({type(e).__name__})")

Connecte a Base (chain_id=8453)
Gas price: 0.0060 gwei
Bloc: 55,084,326

Cout d'un deploiement simple (300,000 gas), par chaine :


  Base: chain_id=8453, gas=0.0060 gwei, cout=0.0000018000 ETH (~$0.0045)
  Polygon: chain_id=137, gas=71.1700 gwei, cout=0.0213510036 ETH (~$0.0107)


  Ethereum L1: chain_id=1, gas=0.0521 gwei, cout=0.0000156431 ETH (~$0.0391)


### Interpretation : Connexion au reseau Base mainnet et cout de deploiement par chaine

**Sortie obtenue** : `Connecte a Base (chain_id=8453)`, `Gas price: 0.0060 gwei` et `Bloc: 55,084,326` — la connexion au reseau Base mainnet reussit : le noeud affiche le chain_id, le gas price et le numero de bloc courant. Les trois lignes suivantes mesurent le **meme** deploiement (300 000 gas) sur trois chaines, a l'instant de l'execution.

| Chaine | chain_id | gas price mesure | Cout (300 000 gas) |
|--------|----------|------------------|--------------------|
| Base | 8453 | 0.0060 gwei | 0.0000018000 ETH (~$0.0045) |
| Polygon PoS | 137 | 71.1700 gwei | 0.0213510036 ETH (~$0.0107) |
| Ethereum L1 | 1 | 0.0521 gwei | 0.0000156431 ETH (~$0.0391) |

**Points cles** :
1. **Le cout se mesure, il ne se devine pas.** Sur la mesure affichee, un deploiement simple coute ~$0.0045 sur Base, ~$0.0107 sur Polygon PoS et ~$0.0391 sur Ethereum L1. La base fee d'Ethereum L1 bouge de plusieurs ordres de grandeur d'un jour a l'autre : c'est pourquoi aucun montant n'est fige dans ce carnet.
2. **Deux modeles de securite.** Base est un rollup optimiste : sa securite est **bornee** par Ethereum L1 (transactions publiees et prouvees sur L1). Polygon PoS est une **sidechain** : son comite de validateurs est autonome, elle **n'herite pas** de la securite d'Ethereum.
3. Le RPC public `mainnet.base.org` est gratuit mais peut etre lent en production

> **Note technique** : En production, utilisez un RPC payant (Alchemy, Infura) pour une meilleure latence et fiabilite.

***

## 2. Estimation du cout

Avant de deployer, estimons le cout en ETH et en USD.

In [3]:
try:
    from eth_account import Account
    ETH_ACCOUNT_AVAILABLE = True
except ImportError:
    ETH_ACCOUNT_AVAILABLE = False

if WEB3_AVAILABLE and ETH_ACCOUNT_AVAILABLE and PRIVATE_KEY and w3.is_connected():
    account = Account.from_key(PRIVATE_KEY)
    balance = w3.eth.get_balance(account.address)
    balance_eth = w3.from_wei(balance, 'ether')

    # Estimation du cout de deploiement
    estimated_gas = 300000  # Gas typique pour un contrat simple
    gas_price = w3.eth.gas_price
    estimated_cost = estimated_gas * gas_price
    cost_eth = w3.from_wei(estimated_cost, 'ether')

    print(f"Deployer: {account.address}")
    print(f"Balance: {balance_eth:.6f} ETH")
    print(f"\nEstimation deploiement:")
    print(f"  Gas estime: {estimated_gas:,}")
    print(f"  Gas price: {w3.from_wei(gas_price, 'gwei'):.4f} gwei")
    print(f"  Cout estime: {cost_eth:.6f} ETH")
    # 4 decimales : un cout de deploiement sur Base vaut quelques millimes de dollar,
    # et deux decimales l'afficheraient 0,00 -- le nombre affiche ne mesurerait plus son objet
    print(f"  (~${float(cost_eth) * 2500:.4f} a $2500/ETH)")

    if balance < estimated_cost:
        print(f"\nBalance insuffisante. Envoyez du ETH sur Base a:")
        print(f"  {account.address}")
else:
    print("Configurez DEPLOYER_PRIVATE_KEY dans .env")

Configurez DEPLOYER_PRIVATE_KEY dans .env


### Interpretation : Estimation du cout de deploiement

**Sortie obtenue** : `Configurez DEPLOYER_PRIVATE_KEY dans .env` — aucun calcul de cout n'a pu etre effectué, la clé privée n'étant pas configurée dans l'environnement.

**Analyse de la logique d'estimation** :

| Élément | Valeur typique | Description |
|---------|----------------|-------------|
| Gas estimé | 300,000 | Gas pour un contrat simple (variable selon la complexité) |
| Gas price | ~0.0060 gwei | Prix du gas sur Base (variable selon la congestion) |
| Coût estimé | ~0.0000018 ETH | Conversion en ETH (gas × gas_price) |
| Coût USD | ~$0.0045 | À $2500/ETH (environ 0,5 centime) |

**Points clés** :
1. Le coût de déploiement sur Base est **négligeable** : la section 1 mesure le même déploiement sur Ethereum L1 et sur Polygon PoS — c'est cette mesure, et non un montant figé, qui donne l'échelle.
2. L'estimation inclut : gas de création + gas d'exécution du constructor
3. Le coût réel peut varier selon la congestion du réseau

> **Note de sécurité** : Ne commettez JAMAIS de clé privée dans un notebook. Utilisez toujours un fichier `.env` ajouté à `.gitignore`.

***

## 3. Deploiement sur Base

Le processus est identique a Sepolia, seul le chain_id change.

Déploiement du contrat sur le réseau Base mainnet après vérification du solde et de la connexion.

In [4]:
STORAGE_CONTRACT = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title SimpleStorage - Premier contrat deploye sur mainnet
/// @notice Stocke une valeur avec historique des modifications
contract SimpleStorage {
    uint256 public value;
    address public owner;
    uint256 public updateCount;
    
    event Updated(uint256 indexed newValue, address indexed by, uint256 count);
    
    constructor(uint256 _initial) {
        value = _initial;
        owner = msg.sender;
    }
    
    function set(uint256 _v) external {
        require(msg.sender == owner, "SimpleStorage: caller is not the owner");
        value = _v;
        updateCount++;
        emit Updated(_v, msg.sender, updateCount);
    }
}
"""

if WEB3_AVAILABLE:
    # Compiler
    SOLC_VERSION = "0.8.28"
    installed = [str(v) for v in solcx.get_installed_solc_versions()]
    if SOLC_VERSION not in installed:
        solcx.install_solc(SOLC_VERSION)
    solcx.set_solc_version(SOLC_VERSION)

    compiled = solcx.compile_source(
        STORAGE_CONTRACT, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    cid, ci = compiled.popitem()
    print(f"Compile: {cid}, bytecode={len(ci['bin'])} chars")
else:
    print("Section sautee : web3/solcx non disponible")

Compile: <stdin>:SimpleStorage, bytecode=2552 chars


### Interprétation : Compilation du contrat SimpleStorage

**Sortie obtenue** : `Compile: <stdin>:SimpleStorage, bytecode=2552 chars` — le contrat est compilé avec succès par solc 0.8.28, et le bytecode produit fait 2552 caractères hexadécimaux.

| Élément | Valeur | Signification |
|---------|--------|---------------|
| Contract ID | `<stdin>:SimpleStorage` | Identifiant du contrat compilé |
| Bytecode length | 2552 caractères | Taille du bytecode **de création** (celui qui part sur la chaîne) |
| Compilation status | SUCCESS | Le contrat Solidity est valide |

**Analyse du contrat** :
- **Pragma** : `^0.8.28` - Version du compilateur Solidity
- **Variables d'état** : `value`, `owner`, `updateCount`
- **Événements** : `Updated` - log des modifications avec metadata
- **Constructor** : Initialise la valeur et le propriétaire
- **Droits d'accès** : `set()` est protégé par `require(msg.sender == owner, ...)` — seul le propriétaire peut changer la valeur ; sans cette garde, n'importe quelle adresse pourrait écraser l'état du contrat

> **Note technique** : le bytecode de création est celui qui est envoyé sur la chaîne ; une fois déployé, le contrat exécute son **runtime**, plus court que le bytecode de création. Son coût en gas dépend de sa taille : plus le contrat est complexe, plus il coûte cher à déployer.

**Indice : Exercice de deploiement ERC-20**

Pour deployer un token ERC-20 sur Base mainnet :

**Etape 1 - Preparation du contrat**

Ce contrat n'a **aucune dependance** : il se compile avec l'outil de la section 3 (`solcx.compile_source`), sans bibliotheque externe ni remapping.

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title MyToken - ERC-20 minimal ecrit a la main
contract MyToken {
    mapping(address => uint256) public balanceOf;
    uint256 public totalSupply;
    string public name = "MyToken";
    string public symbol = "MTK";
    uint8 public constant decimals = 18;

    event Transfer(address indexed from, address indexed to, uint256 value);

    constructor(uint256 supply) {
        totalSupply = supply * 10 ** decimals;
        balanceOf[msg.sender] = totalSupply;
        emit Transfer(address(0), msg.sender, totalSupply);
    }

    function transfer(address to, uint256 amount) external returns (bool) {
        require(balanceOf[msg.sender] >= amount, "solde insuffisant");
        balanceOf[msg.sender] -= amount;
        balanceOf[to] += amount;
        emit Transfer(msg.sender, to, amount);
        return true;
    }
}
```

> **Variante** : la version OpenZeppelin du carnet SC-07 importe `@openzeppelin/contracts/token/ERC20/ERC20.sol` ; elle suppose `forge` et le repertoire `foundry-lib/` de la serie (voir `forge_helper.py`), **pas** `solcx.compile_source`. Une dependance d'import se resout par le chemin de compilation, pas par la version du compilateur.

**Etape 2 - Estimation du gas**
- Un token ERC-20 simple coute environ 800 000 a 1 200 000 gas (le double du contrat minimal de la section 3)
- Le cout se calcule avec le gas price **mesure en section 1** : `cout = gas x gas_price` (sur Base, le gas price affiche en section 1 rend ce cout de l'ordre de quelques millimes de dollar)

**Etape 3 - Verification**
- Allez sur BaseScan : `https://basescan.org/address/<VOTRE_ADRESSE>`
- Cliquez sur "Verify and Publish"
- Selectionnez "Solidity (Single file)" : le contrat de l'Etape 1 n'a aucun import
- Copiez le code source et la version du compilateur

**Etape 4 - Comparaison**
- Cout estime vs cout reel (notez la difference)
- Temps de confirmation sur Base (~2 secondes)

> **Rappel** : Gardez votre cle privee secrete et ne la partagez jamais.

In [5]:
# Deploy sur Base mainnet
if WEB3_AVAILABLE and PRIVATE_KEY and w3.is_connected() and ETH_ACCOUNT_AVAILABLE:
    balance = w3.eth.get_balance(account.address)
    estimated_cost = 500000 * w3.eth.gas_price

    if balance < estimated_cost:
        print("DEPLOIEMENT MAINNET - FONDS INSUFFISANTS")
        print("=" * 60)
        print(f"  Balance  : {w3.from_wei(balance, 'ether'):.6f} ETH")
        print(f"  Besoin   : {w3.from_wei(estimated_cost, 'ether'):.6f} ETH")
        print(f"\nEnvoyez du ETH sur Base a : {account.address}")
        print("(Utilisez un bridge L1->L2 ou achetez directement sur Base)")
    else:
        Contract = w3.eth.contract(abi=ci["abi"], bytecode=ci["bin"])

        nonce = w3.eth.get_transaction_count(account.address)
        tx = Contract.constructor(42).build_transaction({
            "from": account.address,
            "nonce": nonce,
            "gas": 500000,
            "gasPrice": w3.eth.gas_price,
            "chainId": w3.eth.chain_id,
        })

        # POINT DE NON-RETOUR: cette transaction coute du vrai ETH
        print("Deploiement en cours (gas reel)...")
        signed = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
        tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
        print(f"TX: {tx_hash.hex()}")

        receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=60)
        actual_cost = receipt.gasUsed * receipt.effectiveGasPrice

        print(f"\nDeploye a: {receipt.contractAddress}")
        print(f"Gas utilise: {receipt.gasUsed:,}")
        print(f"Cout reel: {w3.from_wei(actual_cost, 'ether'):.6f} ETH")
        print(f"Explorer: https://basescan.org/address/{receipt.contractAddress}")
else:
    print("Deploiement non disponible (cle/connexion manquante)")


Deploiement non disponible (cle/connexion manquante)


### Interprétation : Tentative de déploiement mainnet

**Sortie obtenue** : `Deploiement non disponible (cle/connexion manquante)`

**Comportement attendu si la clé était configurée** :

| Étape | Action | Résultat |
|-------|--------|----------|
| 1 | Vérification du solde | Balance ≥ coût estimé |
| 2 | Construction de la transaction | Nonce, gas limit, gas price, chainId |
| 3 | Signature de la transaction | Signature ECDSA avec la clé privée |
| 4 | Envoi de la transaction | `tx_hash` retourné par le RPC |
| 5 | Attente de la confirmation | Receipt avec `contractAddress` |
| 6 | Calcul du coût réel | `gasUsed × effectiveGasPrice` |

**Points clés** :
1. Le déploiement mainnet est **irréversible** - une fois déployé, le code ne peut plus être modifié
2. Le coût réel peut différer de l'estimation (gas price variable)
3. L'adresse du contrat est déterministe (créateur + nonce)

> **Avertissement** : Ne déployez JAMAIS sur mainnet sans avoir testé sur testnet et audité le code.

***

## 4. Verification sur l'explorateur

Après deploiement, votre contrat est:
- **Permanent** : il existera tant que Base existera
- **Public** : n'importe qui peut lire le code et interagir
- **Immuable** : le code ne peut plus etre modifie

Vous pouvez verifier le contrat sur [BaseScan](https://basescan.org/) pour que d'autres puissent lire le code source.

***

## 5. Securite mainnet

### Checklist avant deploiement mainnet

1. **Audit du code** : Tous les tests passent (SC-12 a SC-14)
2. **Testnet d'abord** : Deploye et teste sur Sepolia (SC-24)
3. **Gestion des cles** : Jamais de cle privee en clair dans le code
4. **Gas estimation** : Verifier le cout avant de signer
5. **Proxy pattern** : Si le contrat doit etre evolutif, utiliser un proxy
6. **Multi-sig** : Pour les fonds importants, utiliser un wallet multi-signature

### Erreurs communes
- Deployer avec une cle de test sur mainnet
- Oublier de verifier le code sur l'explorateur
- Ne pas tester sur testnet avant mainnet

***

## Exercice

Deployez un token ERC-20 minimal sur Base (ou Polygon).
Estimez le cout avant deploiement, puis comparez avec le cout reel.

In [6]:
# Exercice : Deploy ERC-20 sur L2 mainnet
# 1. Reprendre le contrat de l'indice (cellule precedente) : version autonome sans dependance,
#    ou version OpenZeppelin de SC-07 via forge_helper (celle-la suppose `forge`, pas solcx)
# 2. Estimer le gas
# 3. Deployer
# 4. Verifier sur BaseScan/PolygonScan
# 5. Comparer cout estime vs reel

pass  # TODO: Completez cet exercice
print("Exercice a completer")

Exercice a completer


### Exercice 2 : Comparer le cout de deploiement entre deux L2

Comparez concretement le cout estime d'un même deploiement sur Base et Polygon
en lisant uniquement le gas price (AUCUNE transaction envoyee).

**Indice** : `w3.eth.gas_price` pour lire le prix du gas. Cout = `300000 * gas_price`.


In [7]:
# Exercice 2 : Comparer le cout de deploiement entre deux L2
# TODO etudiant : comparer le cout estime d'un deploiement sur Base et Polygon
# Etape 1 : Definir RPCS = {"Base": "...", "Polygon": "..."}
# Etape 2 : Pour chaque reseau, se connecter et lire w3.eth.gas_price
# Etape 3 : Estimer 300000 gas, afficher en token natif et en USD
# Les RPC publics doivent repondre a eth_chainId : un endpoint mort leve une exception
# (c'est le cas de https://polygon-rpc.com, qui repond 401 et non un chainId)
if WEB3_AVAILABLE:
    RPCS = {"Base": "https://mainnet.base.org", "Polygon": "https://polygon-bor-rpc.publicnode.com"}
    PRIX_USD = {"Base": 2500.0, "Polygon": 0.50}
    # for reseau, rpc in RPCS.items():
    #     try:
    #         w3_l2 = Web3(Web3.HTTPProvider(rpc))
    #         gp = w3_l2.eth.gas_price
    #         cout = w3_l2.from_wei(300000 * gp, 'ether')
    #         print(f"{reseau}: {cout:.8f} natif (~${float(cout) * PRIX_USD[reseau]:.4f})")
    #     except Exception as e:
    #         print(f"{reseau}: connexion echouee ({e})")
    pass  # TODO etudiant
    print("Exercice a completer")
else:
    print("web3 non disponible")
    print("Exercice a completer")

Exercice a completer


### Exercice 3 : Verifier le chain_id avant deploiement

Une erreur classique est de deployer sur le mauvais reseau. Ecrivez un garde-fou
qui verifie le chain_id avant tout deploiement.

**Indice** : Base=8453, Sepolia=11155111, Polygon=137, Ethereum=1.


In [8]:
# Exercice 3 : Garde-fou de chain_id avant deploiement
# TODO etudiant : verifier qu'on est connecte au bon reseau avant deploiement
# Etape 1 : Definir EXPECTED_CHAIN_ID = 8453 (Base mainnet)
# Etape 2 : Implementer verifier_reseau(w3, expected) -> bool
# Etape 3 : Tester sur la connexion Base actuelle
EXPECTED_CHAIN_ID = 8453

def verifier_reseau(w3, expected):
    # TODO etudiant : retourner True si w3.eth.chain_id == expected
    return None  # TODO etudiant

if WEB3_AVAILABLE and w3.is_connected():
    # ok = verifier_reseau(w3, EXPECTED_CHAIN_ID)
    # print("Reseau valide pour deploiement :", ok)
    print("Exercice a completer")
else:
    print("Connexion Base requise (section 1)")
    print("Exercice a completer")


Exercice a completer

## Resume et perspectives

Ce carnet a couvert le passage du testnet au deploiement mainnet : la configuration d'un RPC public sur Base, la **mesure** du cout d'un deploiement sur trois chaines (Base, Polygon PoS, Ethereum L1) a l'instant de l'execution, la compilation d'un contrat Solidity avec `py-solc-x` et son garde-fou d'acces, les precautions a verifier avant de signer, et les trois exercices (ERC-20 minimal, comparaison de cout entre deux chaines, garde-fou de `chain_id`).

**Ce que le carnet a mesure, et ce qu'il n'a pas fait** : les sorties montrent la connexion reussie, la compilation reussie et la logique d'estimation ; le **deploiement lui-meme n'a pas ete execute**, faute de cle privee configuree dans l'environnement (`DEPLOYER_PRIVATE_KEY`). La comparaison « cout estime / cout reel » reste donc a faire par l'apprenant, sur une machine ou la cle est configuree. La verification du code source sur l'explorateur (BaseScan) est de meme une **etape manuelle** : le carnet affiche l'adresse du contrat, il n'appelle pas l'API de l'explorateur.

**Deux familles de securite, pas une seule** : Base est un rollup optimiste dont la securite est **bornee** par Ethereum L1 ; Polygon PoS est une **sidechain** EVM, avec son propre comite de validateurs, et n'herite pas de la securite d'Ethereum. Le cout de deploiement se mesure en section 1, il ne se lit pas dans un tableau fige.

Le carnet suivant, [SC-26-Final-Project-Python](SC-26-Final-Project-Python.ipynb), propose un projet capstone integrant l'ensemble des concepts de la serie : smart contract de gouvernance, chiffrement homomorphique, preuves ZKP et deploiement complet.

**Navigation** : [Sommaire](../README.md) | [<< Précédent](SC-24-Testnet-Deploy-Python.ipynb) | [Suivant >>](SC-26-Final-Project-Python.ipynb)

[Retour au sommaire](../README.md)